# CropDoc: AI-Powered Plant Disease Detection
## Kaggle Training Notebook

**Team:** Hamza Shahid, Hottam ud din, Tariq Jamil  
**Supervisor:** Dr. Bahar Ali  
**Institution:** Institute of Management Sciences, Peshawar  
**Program:** BS Data Science, Session 2020–2024

---
### Table of Contents
1. Setup & Imports
2. Dataset Loading
3. EDA: Class Distribution & Sample Images
4. Preprocessing & Data Loaders
5. ResNet-9 Architecture
6. Training Loop
7. Model Checkpoint & ONNX Export
8. Grad-CAM Visualisation
9. Evaluation: Confusion Matrix & Per-Class Accuracy

---
## Section 1 — Setup & Imports

In [ ]:
# Install extra dependency
!pip install torchsummary -q

# ── Standard library ──────────────────────────────────────────────────────────
import os
import random
import warnings
warnings.filterwarnings('ignore')

# ── Scientific computing ──────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ── PyTorch core ──────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── torchvision ───────────────────────────────────────────────────────────────
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# ── Image processing ──────────────────────────────────────────────────────────
import cv2
from PIL import Image

# ── Evaluation ────────────────────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix, classification_report

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

---
## Section 2 — Dataset Loading

In [ ]:
# ── Dataset paths (Kaggle environment) ───────────────────────────────────────
DATA_DIR  = '/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)'
TRAIN_DIR = DATA_DIR + '/train'
VALID_DIR = DATA_DIR + '/valid'
TEST_DIR  = '/kaggle/input/new-plant-diseases-dataset/test/test'

# Load with ToTensor to inspect class list (transforms replaced later)
_raw_train = ImageFolder(TRAIN_DIR, transform=transforms.ToTensor())
_raw_valid = ImageFolder(VALID_DIR, transform=transforms.ToTensor())

NUM_CLASSES  = len(_raw_train.classes)
CLASS_NAMES  = _raw_train.classes
CLASS_TO_IDX = _raw_train.class_to_idx

print(f'Number of classes : {NUM_CLASSES}')
print(f'Training images   : {len(_raw_train)}')
print(f'Validation images : {len(_raw_valid)}')
print('\nFirst 5 classes   :', CLASS_NAMES[:5])

---
## Section 3 — EDA: Class Distribution & Sample Images

In [ ]:
# ── Count images per class ────────────────────────────────────────────────────
class_counts = {cls: len(os.listdir(os.path.join(TRAIN_DIR, cls)))
                for cls in CLASS_NAMES}

# ── Bar chart ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(22, 6))
bars = ax.bar(range(NUM_CLASSES), list(class_counts.values()),
              color='steelblue', edgecolor='white', linewidth=0.5)
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(
    [c.replace('___', '\n').replace('_', ' ') for c in CLASS_NAMES],
    rotation=90, fontsize=6
)
ax.set_xlabel('Disease Class', fontsize=12)
ax.set_ylabel('Number of Training Images', fontsize=12)
ax.set_title('CropDoc — Training Image Count per Disease Class', fontsize=14, fontweight='bold')
ax.axhline(y=np.mean(list(class_counts.values())), color='red',
           linestyle='--', label=f'Mean = {np.mean(list(class_counts.values())):.0f}')
ax.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /kaggle/working/class_distribution.png')

print(f'\nTotal training images : {sum(class_counts.values()):,}')
print(f'Min images/class      : {min(class_counts.values())} ({min(class_counts, key=class_counts.get)})')
print(f'Max images/class      : {max(class_counts.values())} ({max(class_counts, key=class_counts.get)})')

In [ ]:
# ── 5x5 sample image grid ────────────────────────────────────────────────────
random.seed(RANDOM_SEED)
sample_indices = random.sample(range(len(_raw_train)), 25)

fig, axes = plt.subplots(5, 5, figsize=(16, 16))
fig.suptitle('CropDoc — Sample Training Images (5x5)', fontsize=16, fontweight='bold')

for ax, idx in zip(axes.flatten(), sample_indices):
    img_tensor, label = _raw_train[idx]
    img_np = img_tensor.permute(1, 2, 0).numpy()
    ax.imshow(img_np)
    class_display = CLASS_NAMES[label].replace('___', '\n').replace('_', ' ')
    ax.set_title(class_display, fontsize=6, pad=2)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /kaggle/working/sample_images.png')

---
## Section 4 — Preprocessing & Data Loaders

### Training Transforms
| Transform | Purpose |
|---|---|
| `RandomResizedCrop(256, scale=(0.8, 1.0))` | Simulates variable zoom / partial leaf visibility |
| `RandomHorizontalFlip()` | Rotation invariance (leaves have no L/R asymmetry) |
| `RandomRotation(15)` | Accounts for unsteady smartphone angle in the field |
| `ColorJitter(...)` | Simulates different lighting conditions and cameras |
| `Normalize(ImageNet stats)` | Standardises pixel scale for stable gradient flow |

### Validation Transforms
No augmentation — deterministic crop for consistent evaluation.

In [ ]:
# ImageNet normalisation statistics
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Training transforms (augmented) ──────────────────────────────────────────
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(256, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ── Validation / inference transforms (deterministic) ────────────────────────
valid_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(256),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ── Datasets ─────────────────────────────────────────────────────────────────
train_ds = ImageFolder(TRAIN_DIR, transform=train_tfms)
valid_ds = ImageFolder(VALID_DIR, transform=valid_tfms)

# ── Data loaders ─────────────────────────────────────────────────────────────
BATCH_SIZE  = 32
NUM_WORKERS = 2

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Training batches   : {len(train_loader)}')
print(f'Validation batches : {len(valid_loader)}')

# Verify a batch shape
imgs, labels = next(iter(train_loader))
print(f'Batch image shape  : {imgs.shape}  (B, C, H, W)')
print(f'Batch label shape  : {labels.shape}')

---
## Section 5 — ResNet-9 Architecture

```
Input (3, 256, 256)
  conv1  → (64,  256, 256)
  conv2  → (128,  64,  64)  [MaxPool4]
  res1   → (128,  64,  64)  [skip]
  conv3  → (256,  16,  16)  [MaxPool4]
  conv4  → (512,   4,   4)  [MaxPool4]
  res2   → (512,   4,   4)  [skip]  ← Grad-CAM target
  MaxPool4 → Flatten → (512,)
  Dropout(0.5) → Linear(512, 38)
```
Total parameters: ~6.6 M

In [ ]:
def conv_block(in_channels: int, out_channels: int, pool: bool = False) -> nn.Sequential:
    """Conv2d + BatchNorm2d + ReLU, with optional MaxPool2d(4)."""
    layers = [
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    ]
    if pool:
        layers.append(nn.MaxPool2d(4))
    return nn.Sequential(*layers)


class ResNet9(nn.Module):
    """Lightweight 9-layer residual network for 38-class plant disease classification."""

    def __init__(self, in_channels: int = 3, num_classes: int = 38):
        super().__init__()

        # Feature extraction
        self.conv1 = conv_block(in_channels, 64)            # (64, 256, 256)
        self.conv2 = conv_block(64, 128, pool=True)         # (128, 64, 64)
        self.res1  = nn.Sequential(                          # skip connection
            conv_block(128, 128),
            conv_block(128, 128),
        )                                                    # (128, 64, 64)

        self.conv3 = conv_block(128, 256, pool=True)        # (256, 16, 16)
        self.conv4 = conv_block(256, 512, pool=True)        # (512, 4, 4)
        self.res2  = nn.Sequential(                          # skip connection — Grad-CAM target
            conv_block(512, 512),
            conv_block(512, 512),
        )                                                    # (512, 4, 4)

        # Classifier head
        self.classifier = nn.Sequential(
            nn.MaxPool2d(4),     # (512, 1, 1)
            nn.Flatten(),        # (512,)
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, xb: torch.Tensor) -> torch.Tensor:
        out = self.conv1(xb)
        out = self.conv2(out)
        out = self.res1(out) + out   # residual 1
        out = self.conv3(out)
        out = self.conv4(out)
        out = self.res2(out) + out   # residual 2
        return self.classifier(out)


# Instantiate and move to device
model = ResNet9(in_channels=3, num_classes=NUM_CLASSES).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params:,}')
print(model)

---
## Section 6 — Training Loop

**Configuration:**
- Optimizer: Adam (`weight_decay=1e-4`)
- Scheduler: OneCycleLR (`max_lr=0.01`, 20 epochs)
- Gradient clipping: `0.1`
- Epochs: 20
- Best checkpoint saved to `/kaggle/working/cropdoc_resnet9.pth`

In [ ]:
def accuracy(outputs: torch.Tensor, labels: torch.Tensor) -> float:
    """Top-1 accuracy as a Python float."""
    _, preds = torch.max(outputs, dim=1)
    return (preds == labels).float().mean().item()


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader):
    """Run one validation pass; return avg loss and accuracy."""
    model.eval()
    losses, accs = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        loss = F.cross_entropy(out, labels)
        losses.append(loss.item())
        accs.append(accuracy(out, labels))
    return float(np.mean(losses)), float(np.mean(accs))


def get_lr(optimizer: torch.optim.Optimizer) -> float:
    for pg in optimizer.param_groups:
        return pg['lr']

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
EPOCHS      = 20
MAX_LR      = 0.01
WEIGHT_DECAY = 1e-4
GRAD_CLIP   = 0.1
CHECKPOINT  = '/kaggle/working/cropdoc_resnet9.pth'

# ── Optimizer and scheduler ───────────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=MAX_LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=MAX_LR,
    epochs=EPOCHS, steps_per_epoch=len(train_loader)
)

# ── Training loop ─────────────────────────────────────────────────────────────
history = []
best_val_acc = 0.0

for epoch in range(EPOCHS):
    # ── Training phase ────────────────────────────────────────────────────────
    model.train()
    train_losses, lrs = [], []

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        out  = model(imgs)
        loss = F.cross_entropy(out, labels)
        loss.backward()

        # Gradient clipping
        nn.utils.clip_grad_value_(model.parameters(), GRAD_CLIP)

        optimizer.step()
        scheduler.step()

        train_losses.append(loss.item())
        lrs.append(get_lr(optimizer))

    # ── Validation phase ──────────────────────────────────────────────────────
    val_loss, val_acc = evaluate(model, valid_loader)
    train_loss = float(np.mean(train_losses))

    record = {
        'epoch'      : epoch + 1,
        'train_loss' : train_loss,
        'val_loss'   : val_loss,
        'val_acc'    : val_acc,
        'last_lr'    : lrs[-1],
    }
    history.append(record)

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}]  "
          f"lr: {lrs[-1]:.5f}  "
          f"train_loss: {train_loss:.4f}  "
          f"val_loss: {val_loss:.4f}  "
          f"val_acc: {val_acc:.4f}")

    # ── Save best checkpoint ──────────────────────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch'           : epoch + 1,
            'model_state_dict': model.state_dict(),
            'class_to_idx'    : CLASS_TO_IDX,
            'val_accuracy'    : best_val_acc * 100.0,
            'train_history'   : history,
        }, CHECKPOINT)
        print(f'  --> Best checkpoint saved (val_acc={best_val_acc:.4f})')

print(f'\nTraining complete. Best val accuracy: {best_val_acc*100:.2f}%')

In [ ]:
# ── Training history plots ─────────────────────────────────────────────────
epochs_range = [r['epoch'] for r in history]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('CropDoc — Training History', fontsize=14, fontweight='bold')

# Loss
axes[0].plot(epochs_range, [r['train_loss'] for r in history], '-bo', label='Train')
axes[0].plot(epochs_range, [r['val_loss']   for r in history], '-ro', label='Validation')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss vs. Epoch'); axes[0].legend(); axes[0].grid(True)

# Accuracy
axes[1].plot(epochs_range, [r['val_acc'] * 100 for r in history], '-go')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Validation Accuracy (%)')
axes[1].set_title('Validation Accuracy vs. Epoch'); axes[1].grid(True)

# Learning rate
axes[2].plot(epochs_range, [r['last_lr'] for r in history], '-mo')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Learning Rate')
axes[2].set_title('LR (end of epoch) vs. Epoch'); axes[2].grid(True)

plt.tight_layout()
plt.savefig('/kaggle/working/training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /kaggle/working/training_history.png')

---
## Section 7 — Model Checkpoint & ONNX Export

In [ ]:
# ── Load best checkpoint for export and evaluation ───────────────────────────
ckpt = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded checkpoint from epoch {ckpt["epoch"]} '
      f'with val_accuracy = {ckpt["val_accuracy"]:.2f}%')

In [ ]:
# ── ONNX export ───────────────────────────────────────────────────────────────
ONNX_PATH  = '/kaggle/working/cropdoc_resnet9.onnx'
dummy_input = torch.randn(1, 3, 256, 256).to(device)

torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    opset_version=11,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
)
print(f'ONNX model exported to: {ONNX_PATH}')
print(f'ONNX file size: {os.path.getsize(ONNX_PATH) / 1e6:.1f} MB')

---
## Section 8 — Grad-CAM Visualisation

Grad-CAM targets `model.res2` — the second residual block.  
Algorithm steps:
1. Forward hook captures activations `A` ∈ ℝ^(C×H×W)
2. Backward hook captures gradients `∂y_c/∂A`
3. Global average pool → channel weights `α_k`
4. Weighted sum + ReLU → `L^c = ReLU(Σ α_k A_k)`
5. Normalise and resize to 256×256

In [ ]:
class GradCAM:
    """Grad-CAM implementation using forward and backward hooks."""

    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model       = model
        self.activations = None
        self.gradients   = None

        self._fwd = target_layer.register_forward_hook(self._save_activation)
        self._bwd = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def remove_hooks(self):
        self._fwd.remove()
        self._bwd.remove()

    def generate(self, input_tensor: torch.Tensor, class_idx: int = None):
        """Return (heatmap [0,1] as numpy H×W, predicted_class_idx)."""
        self.model.zero_grad()
        logits = self.model(input_tensor)

        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()

        logits[0, class_idx].backward()

        # Channel importance weights via global average pooling of gradients
        weights = self.gradients.mean(dim=[0, 2, 3])      # (C,)
        cam     = self.activations[0]                      # (C, H, W)
        for i, w in enumerate(weights):
            cam[i] = cam[i] * w

        cam = cam.mean(dim=0).cpu().numpy()                # (H, W)
        cam = np.maximum(cam, 0)                           # ReLU
        if cam.max() > 0:
            cam /= cam.max()                               # normalise to [0,1]

        return cam, class_idx


def overlay_heatmap(pil_image: Image.Image, cam: np.ndarray, alpha: float = 0.5):
    """Blend Grad-CAM heatmap onto a PIL image; return RGB numpy array."""
    img_rgb = np.array(pil_image.convert('RGB'))
    h, w    = img_rgb.shape[:2]
    cam_rs  = cv2.resize(cam, (w, h))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_rs), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    return np.uint8(img_rgb * (1 - alpha) + heatmap * alpha)


print('GradCAM class defined.')

In [ ]:
# ── Visualise Grad-CAM for 6 sample validation images ─────────────────────
random.seed(RANDOM_SEED)
sample_idx = random.sample(range(len(valid_ds)), 6)

grad_cam = GradCAM(model, target_layer=model.res2)
model.train()  # Enable grad through BatchNorm

fig, axes = plt.subplots(6, 3, figsize=(12, 24))
fig.suptitle('CropDoc — Grad-CAM Visualisations (target layer: model.res2)',
             fontsize=13, fontweight='bold')

# Inverse normalisation for display
inv_mean = np.array(IMAGENET_MEAN)
inv_std  = np.array(IMAGENET_STD)

for row, idx in enumerate(sample_idx):
    tensor, label = valid_ds[idx]
    input_t = tensor.unsqueeze(0).to(device).requires_grad_(True)

    cam, pred_idx = grad_cam.generate(input_t)

    # Denormalise for display
    img_np = tensor.permute(1, 2, 0).numpy()
    img_np = np.clip(img_np * inv_std + inv_mean, 0, 1)
    pil_img = Image.fromarray(np.uint8(img_np * 255))

    heatmap = cv2.applyColorMap(np.uint8(255 * cv2.resize(cam, (256, 256))), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    blended = overlay_heatmap(pil_img, cam)

    pred_name  = CLASS_NAMES[pred_idx].replace('___', ' | ').replace('_', ' ')
    true_name  = CLASS_NAMES[label].replace('___', ' | ').replace('_', ' ')
    correct    = '✓' if pred_idx == label else '✗'

    axes[row, 0].imshow(img_np); axes[row, 0].set_title(f'Original\nTrue: {true_name}', fontsize=7)
    axes[row, 1].imshow(heatmap); axes[row, 1].set_title('Grad-CAM Heatmap', fontsize=7)
    axes[row, 2].imshow(blended); axes[row, 2].set_title(f'{correct} Pred: {pred_name}', fontsize=7)

    for ax in axes[row]:
        ax.axis('off')

grad_cam.remove_hooks()
model.eval()

plt.tight_layout()
plt.savefig('/kaggle/working/gradcam_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /kaggle/working/gradcam_samples.png')

---
## Section 9 — Evaluation: Confusion Matrix, Per-Class Accuracy & Classification Report

In [ ]:
# ── Collect all validation predictions ────────────────────────────────────────
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in valid_loader:
        imgs = imgs.to(device)
        out  = model(imgs)
        _, preds = torch.max(out, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

overall_acc = (all_preds == all_labels).mean() * 100
print(f'Overall validation accuracy: {overall_acc:.2f}%')

In [ ]:
# ── Confusion matrix (seaborn heatmap) ───────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(22, 20))
sns.heatmap(
    cm,
    ax=ax,
    cmap='Blues',
    xticklabels=[c.split('___')[-1].replace('_', ' ') for c in CLASS_NAMES],
    yticklabels=[c.split('___')[-1].replace('_', ' ') for c in CLASS_NAMES],
    fmt='d',
    linewidths=0.3,
    linecolor='gray',
    annot=cm.shape[0] <= 20,   # annotate only if small enough to read
)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('CropDoc — Confusion Matrix (Validation Set)', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=90, labelsize=7)
ax.tick_params(axis='y', rotation=0,  labelsize=7)
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /kaggle/working/confusion_matrix.png')

In [ ]:
# ── Per-class accuracy bar chart ──────────────────────────────────────────────
per_class_acc = []
for cls_idx in range(NUM_CLASSES):
    mask    = all_labels == cls_idx
    acc_cls = (all_preds[mask] == cls_idx).mean() * 100 if mask.sum() > 0 else 0.0
    per_class_acc.append(acc_cls)

per_class_acc = np.array(per_class_acc)
sorted_idx    = np.argsort(per_class_acc)

fig, ax = plt.subplots(figsize=(10, 20))
colors  = ['#d62728' if a < 90 else '#2ca02c' for a in per_class_acc[sorted_idx]]
ax.barh(
    range(NUM_CLASSES),
    per_class_acc[sorted_idx],
    color=colors, edgecolor='white', linewidth=0.3
)
ax.set_yticks(range(NUM_CLASSES))
ax.set_yticklabels(
    [CLASS_NAMES[i].replace('___', ' | ').replace('_', ' ') for i in sorted_idx],
    fontsize=7
)
ax.set_xlabel('Accuracy (%)', fontsize=12)
ax.set_title('CropDoc — Per-Class Accuracy (Validation Set)\nRed = < 90%', fontsize=13, fontweight='bold')
ax.axvline(x=np.mean(per_class_acc), color='blue', linestyle='--',
           label=f'Mean = {np.mean(per_class_acc):.1f}%')
ax.set_xlim(0, 105)
ax.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /kaggle/working/per_class_accuracy.png')

print(f'\nPer-class accuracy stats:')
print(f'  Mean   : {per_class_acc.mean():.2f}%')
print(f'  Std    : {per_class_acc.std():.2f}%')
print(f'  Min    : {per_class_acc.min():.2f}% ({CLASS_NAMES[per_class_acc.argmin()]})')
print(f'  Max    : {per_class_acc.max():.2f}% ({CLASS_NAMES[per_class_acc.argmax()]})')
print(f'  Classes < 90%: {(per_class_acc < 90).sum()}')

In [ ]:
# ── Full classification report ────────────────────────────────────────────────
print('='*80)
print('CROPDOC — CLASSIFICATION REPORT (Validation Set)')
print('='*80)
report = classification_report(
    all_labels, all_preds,
    target_names=[c.replace('___', ' | ') for c in CLASS_NAMES],
    digits=4
)
print(report)

# Save report to text file
with open('/kaggle/working/classification_report.txt', 'w') as f:
    f.write('CROPDOC — CLASSIFICATION REPORT (Validation Set)\n')
    f.write('='*80 + '\n')
    f.write(report)
print('Saved: /kaggle/working/classification_report.txt')

In [ ]:
# ── Final summary ──────────────────────────────────────────────────────────────
print('=' * 60)
print('  CropDoc — Training Complete')
print('=' * 60)
print(f'  Model           : ResNet-9 (from scratch)')
print(f'  Parameters      : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'  Epochs          : {EPOCHS}')
print(f'  Best val acc    : {best_val_acc*100:.2f}%')
print(f'  Checkpoint      : {CHECKPOINT}')
print(f'  ONNX export     : {ONNX_PATH}')
print('=' * 60)
print('Output files saved to /kaggle/working/ :')
for f in sorted(os.listdir('/kaggle/working/')):
    size = os.path.getsize(f'/kaggle/working/{f}')
    print(f'  {f:40s}  {size/1e6:.1f} MB')